In [0]:
!pip install openpyxl

In [0]:
import pandas as pd
import glob
import re
from pyspark.sql.functions import col, datediff, when, concat, coalesce, lit, date_format, to_timestamp, udf,to_date, length, regexp_replace
from pyspark.sql.types import IntegerType
from datetime import timedelta, datetime

In [0]:
def extract_date_from_filename(filename):
    """Extract date from filename in format 'dd.mm.yyyy'."""
    match = re.search(r'(\d{2})\.(\d{2})\.(\d{4})\.xlsx', filename)
    if match:
        day, month, year = match.groups()
        return f'{year}-{month}-{day}'  # Format as YYYY-MM-DD
    return None

def add_reporting_date(file_paths,sht_name):
    # Create an empty list to store DataFrames
    dfs = []
    for file in file_paths:
        # Extract the filename
        filename = file.split('/')[-1]
        
        # Read the Excel file
        df = pd.read_excel(file, sheet_name=sht_name, dtype=str)
        
        # Extract date from filename for all files
        reporting_date = extract_date_from_filename(filename)
        df['Reporting Date'] = reporting_date
        
        dfs.append(df)
    return dfs
    
# Define UDF to calculate working days
def network_days(start_date, end_date):
    if start_date is None or end_date is None:
        return None

    # Define weekend days (Saturday=5, Sunday=6)
    weekend = {5, 6}
    working_days_count = 0

    # Iterate through the date range
    current_date = start_date
    while current_date <= end_date:
        if current_date.weekday() not in weekend:
            working_days_count += 1
        current_date += timedelta(days=1)

    return working_days_count

In [0]:
# main path
source_path = "/dbfs/mnt/stppeedp/ppeedp/landing/data0/staging/eag/ey/ap_automation/"

# Define the path to write the output 
destination_path = 'dbfs:/mnt/stppeedp/ppeedp/prod/eag/ey/fdw/mfr/invoice_turnaround_time'
destination_path_sheet1 = 'dbfs:/mnt/stppeedp/ppeedp/prod/eag/ey/fdw/mfr/invoice_turnaround_time_sheet1'
invoice_turnaround_time_paths = source_path + 'invoice_turnaround_time/INV*.xlsx'
supplier_mapping_path = source_path+'input_files/Supplier Mapping.xlsx'

In [0]:
# Reading Supplier Mapping 
supplier_mapping_df = pd.read_excel(supplier_mapping_path, dtype=str)
supplier_mapping = spark.createDataFrame(supplier_mapping_df)
supplier_mapping = supplier_mapping.withColumnRenamed("Vendor","SP Vendor").withColumnRenamed("Category","SP Category")

In [0]:
# Define the path to your Excel files
file_paths = glob.glob(invoice_turnaround_time_paths)

# Add Reporting Date Column
dfs = add_reporting_date(file_paths,sht_name='Final')

# Concatenate all DataFrames
combined_df = pd.concat(dfs, ignore_index=True)

# Rename column name to avoid error at the time of write
combined_df = combined_df.rename(columns={"Payment block": "Payment_block","Doc.status":"Doc status",
                                          "Ref.key (header) 2":"Ref key (header) 2",
                                          "Ref.key (header) 1":"Ref key (header) 1"})

# Convert to Spark DataFrame
invoice_turnaround_time_df = spark.createDataFrame(combined_df)

# Cast all columns to StringType to avoid error at the time of write
invoice_turnaround_time_df = invoice_turnaround_time_df.select([col(c).cast("string") for c in invoice_turnaround_time_df.columns])

# Register UDF
network_days_udf = udf(network_days, IntegerType())

invoice_turnaround_time_df = invoice_turnaround_time_df.withColumn("Diff", datediff(col("Posting Day"), col("Date of Parking"))).withColumn(
    "Diff Excl Weekends", 
    network_days_udf(to_date(col("Date of Parking")),to_date(col("Posting Day")))
).withColumn(
    "Type", lit(0)
).withColumn(
    "Doc Date - Posting Date", 
    network_days_udf(to_date(col("Document Date")),to_date(col("Posting Day")))
).withColumn(
    "Doc Date - Due Date", 
    network_days_udf(to_date(col("Document Date")),to_date(col("Net due date")))
).withColumn(
    "Entry less Doc Date (invoice issue lag)",
    datediff(col("Date of Parking"), col("Document Date"))
).withColumn(
    "(invoice isue lag - adjusted). Entry date less doc date", 
    when(col("Entry less Doc Date (invoice issue lag)") < 0, 0)\
        .otherwise(col("Entry less Doc Date (invoice issue lag)"))
).withColumn("Invoice issue lag grouping",
             when(col("`(invoice isue lag - adjusted). Entry date less doc date`") <= 7, "7 days")
             .when((col("`(invoice isue lag - adjusted). Entry date less doc date`") > 7) & (col("`(invoice isue lag - adjusted). Entry date less doc date`") <= 14), "8 to 14 days")
             .when((col("`(invoice isue lag - adjusted). Entry date less doc date`") > 14) & (col("`(invoice isue lag - adjusted). Entry date less doc date`") <= 21), "15 to 21 days")
             .otherwise("Above 21 days")
).withColumn("AP Delay ageing",
             when(col("Diff Excl Weekends") <= 2, "2 days")
             .when((col("Diff Excl Weekends") > 2) & (col("Diff Excl Weekends") <= 4), "3 to 4 days")
             .when((col("Diff Excl Weekends") > 4) & (col("Diff Excl Weekends") <= 6), "4 to 5 days")
             .otherwise("Above 6 days")
).withColumn("CC+Vendor", concat(col("Company Code"), col("Vendor"))).drop("AP Manager")

# Drop duplicates to keep only one row per CC_Code_Vendor
supplier_mapping_unique = supplier_mapping.dropDuplicates(["CC Code+Vendor"])

# Join with supplier_mapping_unique to get Vendor_Mapping
invoice_turnaround_time_df=invoice_turnaround_time_df.join(supplier_mapping_unique,invoice_turnaround_time_df["CC+Vendor"]==supplier_mapping_unique["CC Code+Vendor"], "left")\
    .drop(supplier_mapping_unique["Company Code"])\
    .drop("CC Code+Vendor","Team Lead","Vendor type","Vendor Status","CC+Vendor")\
    .withColumnRenamed("Manager","AP Manager")\
    .withColumn("AP Manager",coalesce(col("AP Manager"), lit('Unassigned')))

invoice_turnaround_time_df = invoice_turnaround_time_df.withColumn(
    "Category",
    when(
        (length(col("Vendor")) < 7) & (col("Vendor").rlike("^[0-9]+$")), "Staff Expense"  # Numeric and less than 7 digits
    ).when(
        (length(col("Vendor")) < 7) & (col("Vendor").rlike("^[a-zA-Z0-9]+$")), "ICH NON DOC/ICH DOC"  # Alphanumeric and less than 7 digits
    ).when(
        col("SP Vendor").isNotNull(), col("SP Category")  # Use the lookup value if available
    ).otherwise("Unassigned")  # Default to "Unassigned" if no other conditions are met
).withColumn("Category", coalesce(col("Category"), lit('Unassigned'))).drop("SP Vendor", "SP Category")

# Write the data to the destination
invoice_turnaround_time_df.write.mode("overwrite").parquet(destination_path)
print(f"Data has been successfully processed and written at {destination_path}")

### Sheet1 Processing

In [0]:
# Define the path to your Excel files
file_paths = glob.glob(invoice_turnaround_time_paths)

# Add Reporting Date Column
dfs = add_reporting_date(file_paths,sht_name='Sheet1')

# Concatenate all DataFrames
combined_df = pd.concat(dfs, ignore_index=True)

# Rename column name to avoid error at the time of write
combined_df = combined_df.rename(columns={"Payment block": "Payment_block","Doc.status":"Doc status",
                                          "Ref.key (header) 2":"Ref key (header) 2",
                                          "Ref.key (header) 1":"Ref key (header) 1"})

# Convert to Spark DataFrame
invoice_turnaround_time_df = spark.createDataFrame(combined_df)

# Cast all columns to StringType to avoid error at the time of write
invoice_turnaround_time_df = invoice_turnaround_time_df.select([col(c).cast("string") for c in invoice_turnaround_time_df.columns])


# Write the data to the destination
invoice_turnaround_time_df.write.mode("overwrite").parquet(destination_path_sheet1)
print(f"Data has been successfully processed and written at {destination_path_sheet1}")